In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from pathlib import Path
from core.settings import TradingConfig
from core.contracts import EngineInput, FilterPack
from core.utils import SystemUtils as SU

# --- THE NEW VERTICAL SLICE IMPORTS! ---
from data_pipeline import generate_features
from walk_forward import AlphaEngine, create_walk_forward_analyzer, run_headless_simulation

config = TradingConfig()

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



In [2]:
# notebooks_RLVR_OUTPUT_DIR = Path(
#     r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output"
# )

OUTPUT_DIR = Path(
    r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output"
)

# path to OHLCV, VIX Indices
DATA_DIR = Path(r"c:\Users\ping\Files_win10\python\py311\stocks\data\\")
DATA_PATH_OHLCV = Path(
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet"
)
DATA_PATH_INDICES = Path(
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet"
)

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
# print(f"notebooks_RLVR_OUTPUT_DIR: {notebooks_RLVR_OUTPUT_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output
DATA_DIR: c:\Users\ping\Files_win10\python\py311\stocks\data
DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [3]:
# # === RELOADING FROM PARQUET ===
# features_df = pd.read_parquet(OUTPUT_DIR / "features_df.parquet")
# macro_df = pd.read_parquet(OUTPUT_DIR / "macro_df.parquet")
# df_close_wide = pd.read_parquet(OUTPUT_DIR / "df_close_wide.parquet")
# df_atrp_wide = pd.read_parquet(OUTPUT_DIR / "df_atrp_wide.parquet")
# df_trp_wide = pd.read_parquet(OUTPUT_DIR / "df_trp_wide.parquet")

# df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
# df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
# df_fed = pd.read_parquet(
#     notebooks_RLVR_OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet",
#     engine="pyarrow",
# )

In [4]:
print("Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    # notebooks_RLVR_OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet",
    DATA_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet",
    engine="pyarrow",
)

print("[EXEC] Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    config=config,
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)


# Replace GLOBAL_SETTINGS dictionary access with config attributes
if config.handle_zeros_as_nan:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=config.max_data_gap_ffill)
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)

print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

Loading DataFrames... takes ~6 minutes
[EXEC] Generating Features...
[EXEC] Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
[OK] Pipeline Complete. Tickers: 1578, Days: 16206


In [ ]:
# === SAVE TO PERSISTENCE SANDBOX ===
features_df.to_parquet(OUTPUT_DIR / "features_df.parquet", engine="pyarrow", index=True)
macro_df.to_parquet(OUTPUT_DIR / "macro_df.parquet", engine="pyarrow", index=True)
df_close_wide.to_parquet(
    OUTPUT_DIR / "df_close_wide.parquet", engine="pyarrow", index=True
)
df_atrp_wide.to_parquet(
    OUTPUT_DIR / "df_atrp_wide.parquet", engine="pyarrow", index=True
)
df_trp_wide.to_parquet(OUTPUT_DIR / "df_trp_wide.parquet", engine="pyarrow", index=True)

In [6]:
# # print(f"df_ohlcv.info():\n{df_ohlcv.info()}\n")
# print(f"features_df.info():\n{features_df.info()}\n")

In [7]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [8]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=189,
    holding_period=5,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=200,
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [9]:
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.4f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
----------------------------------------------------------------------
Timeline: [2025-07-16] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (200):
SGOV, SHV, BIL, MINT, USFR, BOXX, PULS, JPST, GTLS, SNDK
JAAA, RVMD, SATS, CIEN, TER, EA, LITE, ASX, VGSH, WDC
GLW, ROIV, SHY, ROST, EWY, WBD, VALE, MU, FTI, ERIC
RIO, VTEB, BE, KEYS, NOK, STX, FIX, UTHR, VCSH, IGSB
MUB, CHRW, EMXC, LRCX, GOOGL, GOOG, HSBC, INTC, TEVA, JNJ
TD, CAT, EWZ, ALB, MKSI, CMI, BNS, BSV, COPX, RY
SU, PBR, EFV, ASML, GSK, EWT, FDX, ONTO, BP, AA
IAG, CVE, SLV, HL, KGC, ITUB, COHR, BHP, AU, JAZZ
IAUM, AMAT, GLDM, SGOL, IAU, APA, B, AZN, XBI, SCCO
EMB, PSLV, EQX, SOXX, SCHF, GLD, VEA, BG, SPDW, NEM
IEMG, VEU, IONS, GDX, EEM, VXUS, PHYS, LSCC, HAL, IXUS
ETR, EWJ, MTZ, OPEN, SMH, FNDX, USO, SOXL, KLAC, GDXJ
MOD, NBIS, FTAI, SIL, MEDP, VLO, MPWR, BK, SPIB, VRT
SHEL, AEM, MBB, TTMI, JBHT, BKR, CNQ, IDEV, DD, GFI
ATI, WELL, FE, JNK, EQIX, NVS, FNV, MTSI, VTR,

,Full,Lookback,Holding
Metric,,,
Group Gain,0.6884,0.6641,0.0028
Benchmark Gain,0.1427,0.1254,0.0053
== Gain Delta,0.5457,0.5387,-0.0026
Group Sharpe,4.0505,3.9996,1.2023
Benchmark Sharpe,1.5411,1.4013,2.3401
== Sharpe Delta,2.5094,2.5983,-1.1379
Group Sharpe (ATRP),0.1108,0.1109,0.0199
Benchmark Sharpe (ATRP),0.0723,0.0662,0.0879
== Sharpe (ATRP) Delta,0.0385,0.0447,-0.0680



Target Reward Signal: 0.002771


In [10]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [11]:
##################################
##################################
##################################

In [12]:
result = SU.map_analyzer(analyzer2)
#

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(196,))
[  2]   [PLOT] benchmark_series (shape=(196,))
[  3]   [CALC] normalized_plot_data (shape=(196, 100))
[  4]   [FILE] tickers (len=100)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [D

In [13]:
# Ways to get result tickers
_tickers = SU.fetch(4, result)
print(f"_tickers: {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers: {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'SATS',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM']

_tickers: ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'SATS', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 'VTEB', 'BE', 'KEYS', 'NOK', 'STX', 'FIX', 'UTHR', 'VCSH', 'IGSB', 'MUB', 'CHRW', 'EMXC', 'LRCX', 'GOOGL', 'GOOG', 'HSBC', 'INTC', 'TEVA', 'JNJ', 'TD', 'CAT', 'EWZ', 'ALB', 'MKSI', 'CMI', 'BNS', 'BSV', 'COPX', 'RY', 'SU', 'PBR', 'EFV', 'ASML', 'GSK', 'EWT', 'FDX', 'ONTO', 'BP', 'AA', 'IAG', 'CVE', 'SLV', 'HL', 'KGC', 'ITUB', 'COHR', 'BHP', 'AU', 'JAZZ', 'IAUM', 'AMAT', 'GLDM', 'SGOL', 'IAU', 'APA', 'B', 'AZN', 'XBI', 'SCCO', 'EMB', 'PSLV', 'EQX', 'SOXX', 'SCHF', 'GLD', 'VEA', 'BG', 'SPDW', 'NEM']

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'SATS',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM']

_tickers: ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'SATS', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 'VTEB', 'BE', 'KEYS', 'NOK', 'STX', 'FIX', 'UTHR', 'VCSH', 'IGSB', 'MUB', 'CHRW', 'EMXC', 'LRCX', 'GOOGL', 'GOOG', 'HSBC', 'INTC', 'TEVA', 'JNJ', 'TD', 'CAT', 'EWZ', 'ALB', 'MKSI', 'CMI', 'BNS', 'BSV', 'COPX', 'RY', 'SU', 'PBR', 'EFV', 'ASML', 'GSK', 'EWT', 'FDX', 'ONTO', 'BP', 'AA', 'IAG', 'CVE', 'SLV', 'HL', 'KGC', 'ITUB', 'COHR', 'BHP', 'AU', 'JAZZ', 'IAUM', 'AMAT', 'GLDM', 'SGOL', 'IAU', 'APA', 'B', 'AZN', 'XBI', 'SCCO', 'EMB', 'PSLV', 'EQX', 'SOXX', 'SCHF', 'GLD', 'VEA', 'BG', 'SPDW', 'NEM']

result[4]["obj"]: ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'SATS', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', '

In [14]:
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/Audit_Verification_Report.xlsx')

In [15]:
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

Creating combined dictionary for 101 ticker(s)
Date range: 2025-07-16 00:00:00 to 2026-04-24 00:00:00
Data retrieved for 101 ticker(s) from 2025-07-16 00:00:00 to 2026-04-24 00:00:00
Total rows: 19796
Date range in data: 2025-07-16 00:00:00 to 2026-04-24 00:00:00
  SGOV: 196 rows
  SHV: 196 rows
  BIL: 196 rows
  MINT: 196 rows
  USFR: 196 rows
  BOXX: 196 rows
  PULS: 196 rows
  JPST: 196 rows
  GTLS: 196 rows
  SNDK: 196 rows
  JAAA: 196 rows
  RVMD: 196 rows
  SATS: 196 rows
  CIEN: 196 rows
  TER: 196 rows
  EA: 196 rows
  LITE: 196 rows
  ASX: 196 rows
  VGSH: 196 rows
  WDC: 196 rows
  GLW: 196 rows
  ROIV: 196 rows
  SHY: 196 rows
  ROST: 196 rows
  EWY: 196 rows
  WBD: 196 rows
  VALE: 196 rows
  MU: 196 rows
  FTI: 196 rows
  ERIC: 196 rows
  RIO: 196 rows
  VTEB: 196 rows
  BE: 196 rows
  KEYS: 196 rows
  NOK: 196 rows
  STX: 196 rows
  FIX: 196 rows
  UTHR: 196 rows
  VCSH: 196 rows
  IGSB: 196 rows
  MUB: 196 rows
  CHRW: 196 rows
  EMXC: 196 rows
  LRCX: 196 rows
  GOOGL: 

WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/last_run_tickers_stacked.csv')

In [16]:
_tickers

['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'SATS',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM']

In [17]:
_tickers = ["NVDA", "GOOG"]
idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

df_ohlcv.loc[idx[_tickers]].copy()  # all dates, all columns

df_ohlcv.loc[
    idx[_tickers, pd.Timestamp("2026-04-01") :], :
]  # from start date, all columns

df_ohlcv.loc[
    idx[_tickers, pd.Timestamp("2026-04-01") :], columns
]  # from start date and specific columns

df_ohlcv.loc[
    idx[_tickers, pd.Timestamp("2026-04-01") : pd.Timestamp("2026-04-10")], :
]  # from start to end dates, all columns

Adj Open  Adj High  Adj Low  Adj Close     Volume
Ticker Date                                                         
NVDA   2026-04-01   176.000   177.370  174.750     175.75  168132000
       2026-04-02   172.180   177.490  171.370     177.39  143143200
       2026-04-06   177.160   177.790  175.760     177.64  107564300
       2026-04-07   175.730   178.230  173.660     178.10  132534900
       2026-04-08   184.500   185.260  180.300     182.08  147732700
       2026-04-09   181.840   184.080  180.620     183.91  116428500
       2026-04-10   184.310   190.000  184.300     188.63  160459500
GOOG   2026-04-01   289.980   297.985  289.470     294.90   24403600
       2026-04-02   288.990   295.900  287.570     294.46   13433400
       2026-04-06   294.700   298.425  293.790     297.66   10121700
       2026-04-07   300.140   304.100  295.425     303.93   17317700
       2026-04-08   317.830   319.390  312.705     314.74   20887500
       2026-04-09   313.190   317.430  309.470     316.37   14606800
       2026-04-10   318.225   319.500  314.540     315.72   11997600

In [18]:
# Show all columns (no truncation)
pd.set_option("display.max_columns", None)

# Widen the display to fit your terminal width
pd.set_option("display.width", 2000)

In [19]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

{'NVDA':             Adj Open  Adj High  Adj Low  Adj Close     Volume       ATR      ATRP       TRP        RSI    Mom_21  Consistency     IR_63   Beta_63     DD_21  AutoCorr_15    Ret_1d  Range_Pos_20  Slope_P_5  Slope_V_5  Convexity  RollingStalePct  RollMedDollarVol  RollingSameVolCount
 Date                                                                                                                                                                                                                                                                                     
 2026-05-01    201.28    203.00   197.12     198.45  128647000  6.278859  0.031639  0.029630  52.983235  0.129161          0.2  0.008291  1.735400 -0.083837     0.217483 -0.005612      0.574241  -0.024105  -0.948598  -0.035833              0.0      3.014699e+10                  0.0
 2026-05-04    199.50    201.73   194.74     198.48  125368100  6.329654  0.031891  0.035218  53.011781  0.118891          0.2  0.039841  1.759

In [20]:
#####################
#####################
#####################

In [21]:
import pathlib

# Set display width to prevent line wrapping
pd.set_option("display.width", 3000)

# Show all columns instead of truncated view
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


def load_latest_finviz_parquet(data_dir: str) -> pd.DataFrame:
    path = pathlib.Path(data_dir)
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"

    files = list(path.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {path.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Usage
df_finviz = load_latest_finviz_parquet(DATA_DIR)
print(f"df_finviz:\n{df_finviz}")

DEBUG: Loading file: 2026-05-22_df_finviz_merged_stocks_etfs.parquet
df_finviz:
         No.                                            Company               Index                  Sector                                  Industry         Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M      P/E  Fwd P/E    PEG      P/S     P/B       P/C    P/FCF    Book/sh    Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %       EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %   Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M   ROA %     ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Per

In [22]:
target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]

In [23]:
import ast

try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    # Extract missing tickers from pandas error message
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Remove missing tickers and retry
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

# print(f'missing_tickers: {missing_tickers}\n')
print(f"result:\n{result}")

result:
      No.                Company               Index                  Sector                        Industry  Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %

In [24]:
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, GLW to GOOG
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 5.5+ KB


In [25]:
# result.info()
result["MktCap AUM, M"]

GLW      167010.0
NOK       86520.0
DELL     192590.0
NVDA    5210990.0
GOOG    4620520.0
Name: MktCap AUM, M, dtype: float64

In [26]:
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [27]:
pd.set_option("display.width", 3000)
print(result)

      No.                Company               Index                  Sector                        Industry  Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Lo

In [28]:
result.to_csv(OUTPUT_DIR / "_trash.csv")